# Validação de Modelos
- Com Holdout vs Sem Holdout
- ASE, MAE, MAPE, RMSE
- BIC, AIC

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import statsmodels as sm
import seaborn as sns
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

In [ ]:
from typing import Callable

def plot_ts_w_ho(series: pd.Series, fitted_values: pd.Series, forecast: pd.Series, holdout: int, title: str) -> None:
    """
    Plota uma série temporal com período de holdout, valores ajustados e previsão.

    Parâmetros:
    series (pd.Series): Os dados originais da série temporal.
    fitted_values (pd.Series): Os valores ajustados pelo modelo.
    forecast (pd.Series): Os valores previstos pelo modelo.
    holdout (int): O número de períodos a serem mantidos para validação.
    title (str): O título do gráfico.
    """
    holdout_start = series.index[-holdout]
    plt.figure(figsize=(15, 8))
    plt.scatter(series.index, series, color='blue', label='Dados reais')
    plt.plot(fitted_values.index, fitted_values, color='green', linestyle='-', label='Ajuste do modelo')
    plt.plot(forecast.index, forecast, color='green', linestyle='--', label='Previsão')
    plt.axvline(holdout_start, color='red', linestyle='--', lw=2)
    plt.axvspan(holdout_start, series.index[-1], color='gray', alpha=0.3)
    plt.xticks(series.index, series.index.strftime('%b %Y'), rotation=45, ha='right', fontsize=8)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

def mse(forecast: pd.Series, real: pd.Series) -> float:
    """
    Calcula o erro quadrático médio entre a previsão e os valores reais.

    Parâmetros:
    forecast (pd.Series): Os valores previstos.
    real (pd.Series): Os valores reais.

    Retorna:
    float: O erro quadrático médio.
    """
    forecast = forecast.flatten()
    real = real.flatten()
    return np.mean((forecast - real) ** 2).flatten()[0]

def ts_model_plot(series: pd.Series, method: Callable, holdout: int, forecast_periods: int, title: str, *args, **kwargs) -> None:
    """
    Ajusta um modelo de série temporal e plota os resultados com período de holdout.

    Parâmetros:
    series (pd.Series): Os dados originais da série temporal.
    method (Callable): O método do modelo de série temporal a ser usado para ajuste.
    holdout (int): O número de períodos a serem mantidos para validação.
    forecast_periods (int): O número de períodos a serem previstos.
    title (str): O título do gráfico.
    *args: Argumentos posicionais adicionais a serem passados para o método do modelo.
    **kwargs: Argumentos nomeados adicionais a serem passados para o método do modelo.

    Retorna:
    Nenhum
    """
    model = method(series[:-holdout], *args, **kwargs).fit()
    fitted_values = model.fittedvalues
    forecast = model.forecast(forecast_periods)
    mse_holdout = mse(forecast.values[:holdout], series.values[-holdout:])
    plot_ts_w_ho(series,fitted_values, forecast, holdout,f'{title} - MSE = {round(mse_holdout,2)}' )

In [ ]:
holdout = 24
forecast_periods = 24
holdout_start = monthly_data.index[-holdout]
fitted_values = monthly_data[:-holdout].shift(1)
forecast = pd.Series([float(monthly_data.iloc[-holdout-1])]*forecast_periods, index=pd.date_range(start=holdout_start, periods=forecast_periods, freq='MS'))

real_values = monthly_data.values[-holdout:]
forecast_values = forecast.values[:holdout]

error = float(mse(forecast_values, real_values))